In [1]:
import os
import pandas as pd
import numpy as np

from utils.modelling import gbm, gbm_nl, jump_diffusion, get_jump_params, get_gbm_params

In [9]:
tickers = pd.read_csv('tickers.txt', header=None).values.flatten()

### creates synthetic data using GBM, GBM_nl, Jump Diffusion etc

In [43]:
def make_synthetic_data(method, tickers):
    syndata_path = f"synthetic_data//{method.__name__}"
    os.makedirs(syndata_path, exist_ok=True)
    
    for folder in ['daily', 'hourly']:
        data_folder = f"data//{folder}"
        os.makedirs(os.path.join(syndata_path,folder), exist_ok=True)
        for ticker in tickers:
            file_path = os.path.join(data_folder, f'{ticker}.csv')
            df = pd.read_csv(file_path)
            
            if method.__name__ in ['gbm', 'gbm_nl']:
                prices = method(*get_gbm_params(df))
            elif method.__name__ == 'jump_diffusion':
                prices = method(*get_gbm_params(df), get_jump_params(df.logReturns))[:-1]
            
            if folder == 'hourly':
                syndata_df = pd.DataFrame({
                    'Datetime':df.Datetime,
                    'Close':prices
                })    
            elif folder == 'daily':
                syndata_df = pd.DataFrame({
                'Date':df.Date,
                'Close':prices
            })       
            save_path = os.path.join(syndata_path, folder, f'{ticker}.csv')
            syndata_df.to_csv(save_path, index=False)
    

In [44]:
for method in [gbm, gbm_nl, jump_diffusion]:
    make_synthetic_data(method, tickers)